# AI Video Clips — Wan2.1, Parallel Across All Kaggle GPUs

No Veo subscription needed. This notebook generates short illustrative video clips with the
free, open-source **Wan2.1-T2V-1.3B** text-to-video model, and spreads the work across every
GPU Kaggle gives the session (2x on `GPU T4 x2`) by running one independent worker process per
GPU in parallel, instead of generating clips one at a time on a single GPU.

**Input:** a `beats.json` list of `{id, prompt, narration, duration_seconds, frames}` objects.
Ekalivan's backend now writes this file automatically at
`Ekalivan/backend/rendered/video/<task_id>/beats.json` for every rendered job — upload that file
here (Add Input > Upload, or just drag it into `/kaggle/working/`) to generate clips that match
a real job's content. If no `beats.json` is found, this notebook falls back to the 6 Heat-unit
example beats already used elsewhere in this project, so it still runs end to end untouched.

**Before running:** Settings (right sidebar) > Accelerator > `GPU T4 x2` (recommended — this is
what makes the parallelism useful; `GPU P100` also works but only has 1 GPU) and Internet > On.

**When finished:** download `clips.zip` from the **Output** tab (right sidebar). Each file is
named `<id>.mp4` — matching the `id` field of the beat it was generated from, so it can be
dropped straight into a clip cache keyed the same way.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS < 1:
    raise RuntimeError(
        "No GPU detected. Settings (right sidebar) > Accelerator > 'GPU T4 x2' or 'GPU P100'. "
        "If already set, you may have hit Kaggle's weekly ~30 free GPU-hours quota."
    )
print(f"Detected {NUM_GPUS} GPU(s):")
for i in range(NUM_GPUS):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

In [ ]:
!pip install -q -U diffusers transformers accelerate bitsandbytes imageio imageio-ffmpeg ftfy

## 2. Load the beats to generate

Looks for an uploaded `beats.json` first (as a Kaggle input dataset, or dropped directly into
`/kaggle/working/`); falls back to the Heat-unit example beats otherwise.

In [ ]:
import glob
import json
import os

_FALLBACK_BEATS = [
    {"id": "intro",
     "prompt": "a bright sun radiating warm glowing heat waves over a green landscape, colorful educational cartoon animation style",
     "narration": "Heat comes from many sources around us.",
     "duration_seconds": 4.0, "frames": 25},
    {"id": "what_is_heat",
     "prompt": "glowing molecules inside a solid cube vibrating and moving faster as heat energy is added, simple educational science animation, colorful",
     "narration": "Heat is the total kinetic energy of an object's molecules.",
     "duration_seconds": 3.0, "frames": 21},
    {"id": "hot_cold_temperature",
     "prompt": "a cartoon thermometer dipped into a glass of hot water and a glass of cold water, red mercury rising and falling, colorful educational animation style",
     "narration": "Temperature tells us precisely how hot or cold something is.",
     "duration_seconds": 3.0, "frames": 21},
    {"id": "heat_vs_temperature",
     "prompt": "side by side comparison of a small steaming cup of tea and a large calm pond, educational infographic cartoon animation style, colorful",
     "narration": "Heat and temperature are related, but they are not the same thing.",
     "duration_seconds": 3.0, "frames": 21},
    {"id": "uses_of_expansion",
     "prompt": "a railway track with small visible gaps between the rails under bright sunshine, simple educational cartoon animation style, engineering diagram feel",
     "narration": "Engineers plan for thermal expansion in railway tracks and bridges.",
     "duration_seconds": 3.0, "frames": 21},
    {"id": "outro",
     "prompt": "a happy cartoon classroom scene with a smiling student surrounded by icons of the sun, a thermometer, and a railway track, colorful warm animation style",
     "narration": "That's a wrap on heat and temperature. See you next time!",
     "duration_seconds": 4.0, "frames": 25},
]


def _find_beats_json() -> str | None:
    candidates = ["/kaggle/working/beats.json", *glob.glob("/kaggle/input/*/beats.json")]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


beats_path = _find_beats_json()
if beats_path:
    with open(beats_path, encoding="utf-8") as f:
        ALL_BEATS = json.load(f)
    print(f"loaded {len(ALL_BEATS)} beats from {beats_path}")
else:
    ALL_BEATS = _FALLBACK_BEATS
    print(f"no beats.json found — using {len(ALL_BEATS)} built-in Heat-unit example beats")
    print("searched: /kaggle/working/beats.json and /kaggle/input/*/beats.json")
    print("actual /kaggle/input contents (for debugging a missing dataset mount):")
    for root, dirs, files in os.walk("/kaggle/input"):
        for name in files:
            print(" ", os.path.join(root, name))

for beat in ALL_BEATS:
    beat.setdefault("frames", 21)
    print(f"  {beat['id']}: {beat['prompt'][:70]}...")

## 3. Worker script

One self-contained process: loads its own copy of Wan2.1 and generates the beats it's given.
Launched once per GPU below, each pinned to a different device via `CUDA_VISIBLE_DEVICES`, so
the `NUM_GPUS` workers run truly in parallel rather than sharing one GPU's queue.

In [ ]:
%%writefile /kaggle/working/wan_worker.py
"""Generates a batch of Wan2.1 clips on whichever single GPU this process can see."""

import argparse
import gc
import json
import os
import sys
import time

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, extra limbs, bad anatomy, "
    "static image, watermark, text, subtitles, worst quality, jpeg artifacts, ugly"
)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--beats-json", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--width", type=int, default=640)
    parser.add_argument("--height", type=int, default=368)
    parser.add_argument("--fps", type=int, default=16)
    parser.add_argument("--steps", type=int, default=30)
    args = parser.parse_args()

    import torch
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.utils import export_to_video
    from transformers import BitsAndBytesConfig, UMT5EncoderModel

    tag = f"[worker pid={os.getpid()}]"

    if not torch.cuda.is_available():
        print(f"{tag} ERROR: no GPU visible to this worker", file=sys.stderr, flush=True)
        sys.exit(1)
    print(f"{tag} using GPU: {torch.cuda.get_device_name(0)}", flush=True)

    with open(args.beats_json, encoding="utf-8") as f:
        beats = json.load(f)
    os.makedirs(args.out_dir, exist_ok=True)

    try:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
        )
        text_encoder = UMT5EncoderModel.from_pretrained(
            MODEL_ID, subfolder="text_encoder", quantization_config=quant_config,
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
        )
        print(f"{tag} text encoder loaded with 4-bit quantization", flush=True)
    except Exception as exc:
        # bitsandbytes' compiled CUDA extension sometimes doesn't match whatever GPU
        # Kaggle actually assigns (seen in practice: "named symbol not found" on an
        # older Pascal-generation GPU) — fall back to the unquantized encoder instead
        # of losing the whole run to an opaque low-level CUDA error.
        print(f"{tag} 4-bit quantized load failed ({exc!r}); retrying without quantization", flush=True)
        gc.collect()
        torch.cuda.empty_cache()
        text_encoder = UMT5EncoderModel.from_pretrained(
            MODEL_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
        )
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(
        MODEL_ID, vae=vae, text_encoder=text_encoder, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    )
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{tag} model loaded, generating {len(beats)} beat(s)", flush=True)

    def generate_with_retry(prompt: str, num_frames: int, width: int, height: int, attempt: int = 0):
        try:
            output = pipe(
                prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
                height=height, width=width, num_frames=num_frames,
                guidance_scale=5.0, num_inference_steps=args.steps,
            )
            return output.frames[0]
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if attempt >= 1:
                raise
            half_w = max(16, (width // 2) // 16 * 16)
            half_h = max(16, (height // 2) // 16 * 16)
            print(f"{tag}  OOM at {width}x{height} - retrying once at {half_w}x{half_h}", flush=True)
            return generate_with_retry(prompt, num_frames, half_w, half_h, attempt + 1)

    for beat in beats:
        out_path = os.path.join(args.out_dir, f"{beat['id']}.mp4")
        if os.path.exists(out_path):
            print(f"{tag} skip (already exists): {beat['id']}", flush=True)
            continue
        frames_n = int(beat.get("frames") or 21)
        t0 = time.time()
        print(f"{tag} generating: {beat['id']} -> {beat['prompt'][:80]}", flush=True)
        video_frames = generate_with_retry(beat["prompt"], frames_n, args.width, args.height)
        export_to_video(video_frames, out_path, fps=args.fps)
        print(f"{tag} saved {beat['id']} in {time.time() - t0:.1f}s", flush=True)
        del video_frames
        gc.collect()
        torch.cuda.empty_cache()

    print(f"{tag} done — {len(beats)} beat(s)", flush=True)


if __name__ == "__main__":
    main()

## 4. Split beats across GPUs and generate in parallel

Round-robins beats across `NUM_GPUS` workers (so a 9-beat job on 2 GPUs runs as two ~5-beat and
~4-beat batches concurrently, not 9 beats one after another) and launches one subprocess per
GPU with `CUDA_VISIBLE_DEVICES` pinned, so each worker owns a GPU exclusively.

In [ ]:
import subprocess
import sys
import threading

OUT_DIR = "/kaggle/working/clips"
os.makedirs(OUT_DIR, exist_ok=True)

chunks = [ALL_BEATS[i::NUM_GPUS] for i in range(NUM_GPUS)]

procs = []
for gpu_id, chunk in enumerate(chunks):
    if not chunk:
        continue
    chunk_path = f"/kaggle/working/beats_gpu{gpu_id}.json"
    with open(chunk_path, "w", encoding="utf-8") as f:
        json.dump(chunk, f)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    proc = subprocess.Popen(
        [sys.executable, "/kaggle/working/wan_worker.py", "--beats-json", chunk_path, "--out-dir", OUT_DIR],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    procs.append((gpu_id, proc))
    print(f"launched worker for GPU {gpu_id}: {len(chunk)} beat(s) -> {[b['id'] for b in chunk]}")


def _stream(gpu_id: int, proc: subprocess.Popen) -> None:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(f"[gpu{gpu_id}] {line}", end="")


threads = [threading.Thread(target=_stream, args=(gid, p), daemon=True) for gid, p in procs]
for t in threads:
    t.start()
for _, p in procs:
    p.wait()
for t in threads:
    t.join()

failed = [gid for gid, p in procs if p.returncode != 0]
if failed:
    raise RuntimeError(f"Worker(s) failed for GPU(s): {failed} — scroll up for the error.")
print("\nall workers finished successfully")

## 5. Verify and package for download

In [ ]:
expected = {beat["id"] for beat in ALL_BEATS}
produced = {os.path.splitext(f)[0] for f in os.listdir(OUT_DIR) if f.endswith(".mp4")}
missing = expected - produced
if missing:
    print(f"WARNING: missing clips for beat id(s): {sorted(missing)}")
else:
    print(f"all {len(expected)} clip(s) generated successfully")

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f"  {f}  ({size_kb:.0f} KB)")

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/clips", "zip", OUT_DIR)
print("Download clips.zip from the Output tab (right sidebar).")
print("Each file inside is named <id>.mp4, matching the beats.json 'id' field.")